# MedSkel AI demo: U-Net vessel segmentation on FIVES

This notebook loads the trained model and shows what it does, step by step.

- It needs the trained checkpoint `runs/unet_fives/best.pt` and the FIVES data at `C:\data\FIVES`.
- The pictures below are **validation** images (taken from the official training set, never used for weight updates).
- The official test set is **not** re-evaluated here. Section 5 only reads the saved result file.
- Research prototype for a university class. Not a medical tool.

**Note:** this committed copy keeps the executed text outputs, but the pictures were removed, because the repository rule is "no dataset images in Git". Run the notebook locally to see them. A full executed copy with pictures is in `outputs/demo_executed_with_images.ipynb` on the machine that ran it.

## 1. Load the model

In [1]:
import csv, json
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt
from medskel_ai.data import list_pairs, make_split, load_image, load_mask, resize_image_bilinear, resize_mask_nearest
from medskel_ai.model import count_parameters
from medskel_ai.predict import load_model, predict_probability, make_overlay
from medskel_ai.evaluate import confusion_counts, metrics_from_counts

RUN = Path('runs/unet_fives')
model, device, threshold = load_model(RUN / 'best.pt')
print('device:', device)
print(f'trainable parameters: {count_parameters(model):,}')
print('threshold chosen on validation:', threshold)

device: cuda
trainable parameters: 7,849,601
threshold chosen on validation: 0.4


## 2. Tensor shapes: one 512 x 512 crop through the network

The output has the same height and width as the input, with **one logit per pixel**.

In [2]:
x = torch.rand(1, 3, 512, 512, device=device)
with torch.no_grad():
    e1 = model.enc1(x); e2 = model.enc2(model.pool(e1)); e3 = model.enc3(model.pool(e2))
    e4 = model.enc4(model.pool(e3)); b = model.bottleneck(model.pool(e4))
    out = model(x)
for name, t in [('input', x), ('enc1', e1), ('enc2', e2), ('enc3', e3), ('enc4', e4), ('bottleneck', b), ('output logits', out)]:
    print(f'{name:14s} {list(t.shape)}')

input          [1, 3, 512, 512]
enc1           [1, 32, 512, 512]
enc2           [1, 64, 256, 256]
enc3           [1, 128, 128, 128]
enc4           [1, 256, 64, 64]
bottleneck     [1, 512, 32, 32]
output logits  [1, 1, 512, 512]


## 3. Training curves (read from the real training log)

In [3]:
history = list(csv.DictReader(open(RUN / 'history.csv')))
ep = [int(h['epoch']) for h in history]
fig, (a, b2) = plt.subplots(1, 2, figsize=(11, 3.8))
a.plot(ep, [float(h['train_loss']) for h in history], label='train loss')
a.plot(ep, [float(h['val_loss']) for h in history], label='validation loss')
a.set_xlabel('epoch'); a.set_ylabel('BCE + soft Dice'); a.legend(); a.grid(alpha=.3)
b2.plot(ep, [float(h['val_dice']) for h in history], color='green'); b2.set_xlabel('epoch'); b2.set_ylabel('validation Dice (fixed crops)'); b2.grid(alpha=.3)
plt.show()
best = max(history, key=lambda h: float(h['val_dice']))
print(f"{len(history)} epochs in {float(history[-1]['minutes']):.1f} min; best epoch {best['epoch']} with validation Dice {float(best['val_dice']):.4f}")

38 epochs in 51.2 min; best epoch 32 with validation Dice 0.8783


## 4. Predict whole validation images

The image is cut into overlapping 512 x 512 tiles, each tile is predicted, and the probabilities are averaged where tiles overlap. One image per disease group (the first validation image of each group; a fixed rule, not hand-picked).

In [4]:
_, val_pairs = make_split(list_pairs(r'C:\data\FIVES', 'train'), seed=42)
chosen = [next(p for p in val_pairs if p['group'] == g) for g in 'ADGN']
for pair in chosen:
    image, mask = load_image(pair['image']), load_mask(pair['mask'])
    prob = predict_probability(model, image, device)
    pred = prob >= threshold
    m = metrics_from_counts(*confusion_counts(torch.from_numpy(pred), torch.from_numpy(mask).bool()))
    s = 512
    panels = [('image', resize_image_bilinear(image, s, s)), ('expert mask', resize_mask_nearest(mask, s, s) * 255),
              ('prediction', resize_mask_nearest(pred.astype(np.uint8), s, s) * 255), ('overlay', resize_image_bilinear(make_overlay(image, pred), s, s))]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
    for ax, (title, panel) in zip(axes, panels):
        ax.imshow(panel, cmap='gray', vmin=0, vmax=255); ax.set_title(title); ax.axis('off')
    plt.show()
    print(pair['id'], '(validation)', ' '.join(f'{k}={v:.3f}' for k, v in m.items()))

train/143_A (validation) dice=0.835 iou=0.716 precision=0.876 recall=0.797 specificity=0.997 accuracy=0.991


train/257_D (validation) dice=0.957 iou=0.917 precision=0.965 recall=0.948 specificity=0.997 accuracy=0.993


train/323_G (validation) dice=0.769 iou=0.625 precision=0.918 recall=0.662 specificity=0.998 accuracy=0.984


train/581_N (validation) dice=0.947 iou=0.899 precision=0.936 recall=0.958 specificity=0.993 accuracy=0.989


## 5. Final numbers (saved files, nothing is recomputed here)

In [5]:
for split in ('val', 'test'):
    f = RUN / f'{split}_metrics.json'
    if not f.exists():
        print(split, ': not evaluated yet'); continue
    r = json.loads(f.read_text())
    print(f"\n{split.upper()}  images={r['images']}  threshold={r['threshold']}")
    print('  overall  ' + '  '.join(f"{k}={v['mean']:.4f}" for k, v in r['per_image'].items()))
    for g, v in r['by_group'].items():
        print(f'  group {g}  ' + '  '.join(f"{k}={v[k]['mean']:.4f}" for k in r['per_image']))
    print('  all-black baseline accuracy:', round(r['all_background_baseline']['accuracy'], 4), '(Dice 0, recall 0)')


VAL  images=88  threshold=0.4
  overall  dice=0.8985  iou=0.8272  precision=0.9178  recall=0.8871  specificity=0.9933  accuracy=0.9865
  group A  dice=0.9107  iou=0.8386  precision=0.9222  recall=0.9055  specificity=0.9936  accuracy=0.9880
  group D  dice=0.8630  iou=0.7840  precision=0.8838  recall=0.8466  specificity=0.9907  accuracy=0.9805
  group G  dice=0.8775  iou=0.7938  precision=0.9212  recall=0.8526  specificity=0.9946  accuracy=0.9876
  group N  dice=0.9429  iou=0.8922  precision=0.9441  recall=0.9435  specificity=0.9942  accuracy=0.9898
  all-black baseline accuracy: 0.9238 (Dice 0, recall 0)

TEST  images=200  threshold=0.4
  overall  dice=0.8748  iou=0.7909  precision=0.9093  recall=0.8560  specificity=0.9934  accuracy=0.9858
  group A  dice=0.9082  iou=0.8361  precision=0.9246  recall=0.8983  specificity=0.9935  accuracy=0.9871
  group D  dice=0.8756  iou=0.7871  precision=0.9250  recall=0.8400  specificity=0.9951  accuracy=0.9862
  group G  dice=0.8281  iou=0.7393  pre